Thermal storage behaviojr

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
df = pd.read_excel("xx.xlsx", index_col=0)


In [ ]:
df["SOC"]=df["soc_mwh"]/70
print(df["soc_mwh"].max())
print(df["SOC"].max())

In [ ]:


# ==============================
# 1. TES geometry
# ==============================
def tes_dimensions_cylinder(volume_m3, ratio_height_diameter=2.0):
    D = ((4 * volume_m3) / (np.pi * ratio_height_diameter)) ** (1/3)
    H = ratio_height_diameter * D
    return D, H


# ==============================
# 2. Hourly TES heat loss
# ==============================
def heat_loss_tes_hourly(
    soc,
    T_hot,
    T_cold,
    volume_m3,
    k_wall,
    wall_thickness_m,
    h_out,
    T_ambient,
    ratio_height_diameter=2.0
):

    
    soc = min(max(soc, 0.0), 1.0)

    
    D, H = tes_dimensions_cylinder(volume_m3, ratio_height_diameter)
    r = D / 2

    # Layer heights
    H_hot = soc * H
    H_cold = (1 - soc) * H

    # Surface areas
    A_top = np.pi * r**2
    A_bottom = np.pi * r**2
    A_side_hot = 2 * np.pi * r * H_hot
    A_side_cold = 2 * np.pi * r * H_cold

    A_hot = A_top + A_side_hot
    A_cold = A_bottom + A_side_cold

    # U-value (conduction + convection)
    U = 1 / (wall_thickness_m / k_wall + 1 / h_out)

    # Heat loss [W]
    Q_hot_W = U * A_hot * max(T_hot - T_ambient, 0)
    Q_cold_W = U * A_cold * max(T_cold - T_ambient, 0)

    Q_total_W = Q_hot_W + Q_cold_W

    # Convert to MWh per hour
    Q_total_MWh = Q_total_W / 1e6

    return Q_total_MWh


# ==============================
# 3. PARAMETERS (EDIT THESE)
# ==============================
volume_m3 = 700                # <-- CHANGE
k_wall = 0.04                  # effective insulation conductivity
wall_thickness_m = 0.05        # insulation thickness
h_out = 8.0                    # outside convection
T_ambient_default = 15.0       # fallback ambient
ratio_height_diameter = 2.0


# ==============================
# 4. APPLY TO DATAFRAME
# ==============================

def compute_tes_loss(row):
    # Use ambient if available, else fallback
    if "T_ambient_C" in row:
        T_amb = row["T_ambient_C"]
    else:
        T_amb = T_ambient_default

    return heat_loss_tes_hourly(
        soc=row["SOC"],
        T_hot=row["Ts_day_C"],
        T_cold=row["Tret_C"],
        volume_m3=volume_m3,
        k_wall=k_wall,
        wall_thickness_m=wall_thickness_m,
        h_out=h_out,
        T_ambient=T_amb,
        ratio_height_diameter=ratio_height_diameter
    )


df["TES_heat_loss_MWh"] = df.apply(compute_tes_loss, axis=1)

# Optional: same value but interpreted as MW
df["TES_heat_loss_MW"] = df["TES_heat_loss_MWh"]


# ==============================
# 5. QUICK CHECK
# ==============================
print(df["TES_heat_loss_MWh"].describe())



plt.figure(figsize=(20, 4))
plt.plot(df.index, df["TES_heat_loss_MWh"])
plt.ylabel("Heat loss [MWh/h]")
plt.xlabel("Time")
plt.title("Hourly TES Heat Loss Over the Year")
plt.grid(True)
plt.show()

print(df['TES_heat_loss_MWh'].sum())

In [ ]:
#print colnames
print(df.columns)
sum_year = df['demand_mw'].sum()
print(df['TES_heat_loss_MWh'].sum()/ sum_year)

In [ ]:
network_A = 14000*0.15*3.1416*2
print(network_A)
print(581/network_A)
heatlosstank = (0.044*60000)/3.6
print(heatlosstank)
print(sum_year)




In [ ]:

def tes_dimensions_cylinder(volume_m3, ratio_height_diameter=2.0):
    """
    Calculate diameter and height of a cylindrical TES tank.

    Parameters
    ----------
    volume_m3 : float
        TES volume in m3.
    ratio_height_diameter : float
        Height-to-diameter ratio H/D.

    Returns
    -------
    D : float
        Diameter in m.
    H : float
        Height in m.
    """
    D = ((4 * volume_m3) / (np.pi * ratio_height_diameter)) ** (1 / 3)
    H = ratio_height_diameter * D
    return D, H


def calculate_tes_heat_loss(
    df,
    tes_size_m3,
    soc_col="SOC",
    T_hot_col="Ts_day_C",
    T_cold_col="Tret_C",
    T_ambient_col=None,
    T_ambient_default=15.0,
    k_wall=0.04,
    wall_thickness_m=0.3,
    h_out=8.0,
    ratio_height_diameter=2.0,
    output_col="TES_heat_loss_MWh",
    add_mw_col=True,
    copy=True,
):
    """
    Calculate hourly thermal energy storage heat loss and annual TES heat loss.

    The TES is represented as a vertical cylindrical tank with a hot and cold
    layer. The hot layer height is based on SOC. Heat loss is calculated from
    the top + hot side wall and bottom + cold side wall.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing SOC and temperature columns.
    tes_size_m3 : float
        TES volume in m3.
    soc_col : str
        Column name for state of charge, between 0 and 1.
    T_hot_col : str
        Column name for hot-layer temperature in °C.
    T_cold_col : str
        Column name for cold-layer temperature in °C.
    T_ambient_col : str or None
        Column name for ambient temperature in °C. If None or missing,
        T_ambient_default is used.
    T_ambient_default : float
        Default ambient temperature in °C.
    k_wall : float
        Effective insulation conductivity in W/m/K.
    wall_thickness_m : float
        Insulation thickness in m.
    h_out : float
        Outside convection coefficient in W/m2/K.
    ratio_height_diameter : float
        Height-to-diameter ratio of the cylindrical tank.
    output_col : str
        Name of the output column for TES heat loss in MWh per hour.
    add_mw_col : bool
        If True, also adds a MW column. For hourly data, MWh/h = MW.
    copy : bool
        If True, returns a copy of df. If False, modifies df directly.

    Returns
    -------
    df_out : pandas.DataFrame
        DataFrame with TES heat loss column added.
    annual_tes_heat_loss_mwh : float
        Annual TES heat loss in MWh/year.
    annual_tes_heat_loss_gj : float
        Annual TES heat loss in GJ/year.
    """

    if copy:
        df_out = df.copy()
    else:
        df_out = df

    required_cols = [soc_col, T_hot_col, T_cold_col]
    missing_cols = [col for col in required_cols if col not in df_out.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in df: {missing_cols}")

    if tes_size_m3 <= 0:
        df_out[output_col] = 0.0
        if add_mw_col:
            df_out[output_col.replace("_MWh", "_MW")] = 0.0
        return df_out, 0.0, 0.0

    # Geometry
    D, H = tes_dimensions_cylinder(tes_size_m3, ratio_height_diameter)
    r = D / 2

    A_top = np.pi * r**2
    A_bottom = np.pi * r**2

    # U-value: conduction through insulation + outside convection
    U = 1 / (wall_thickness_m / k_wall + 1 / h_out)

    # Input arrays
    soc = df_out[soc_col].clip(lower=0.0, upper=1.0)
    T_hot = df_out[T_hot_col]
    T_cold = df_out[T_cold_col]

    if T_ambient_col is not None and T_ambient_col in df_out.columns:
        T_ambient = df_out[T_ambient_col]
    else:
        T_ambient = T_ambient_default

    # Layer heights
    H_hot = soc * H
    H_cold = (1 - soc) * H

    # Surface areas
    A_side_hot = 2 * np.pi * r * H_hot
    A_side_cold = 2 * np.pi * r * H_cold

    A_hot = A_top + A_side_hot
    A_cold = A_bottom + A_side_cold

    # Heat loss [W]
    Q_hot_W = U * A_hot * np.maximum(T_hot - T_ambient, 0)
    Q_cold_W = U * A_cold * np.maximum(T_cold - T_ambient, 0)

    Q_total_W = Q_hot_W + Q_cold_W

    # For hourly data: W -> MW -> MWh per hour
    df_out[output_col] = Q_total_W / 1e6

    if add_mw_col:
        mw_col = output_col.replace("_MWh", "_MW")
        df_out[mw_col] = df_out[output_col]

    annual_tes_heat_loss_mwh = df_out[output_col].sum()
    annual_tes_heat_loss_gj = annual_tes_heat_loss_mwh * 3.6

    return df_out, annual_tes_heat_loss_mwh, annual_tes_heat_loss_gj

In [ ]:
heat_loss_year_test = calculate_tes_heat_loss(
    df=df,
    tes_size_m3=1000,
    soc_col="SOC",
    T_hot_col="Ts_day_C",
    T_cold_col="Tret_C",
    T_ambient_col="T_ambient",
    k_wall=0.04,
    wall_thickness_m=0.3,
    h_out=8.0,
    ratio_height_diameter=2.0
)
#plot hourly heat loss
plt.figure(figsize=(20, 4))
plt.plot(heat_loss_year_test[0].index, heat_loss_year_test[0]["TES_heat_loss_MWh"])
plt.ylabel("Heat loss [MWh/h]") 
plt.xlabel("Time")
plt.title("Hourly TES Heat Loss Over the Year (1000 m3)")
plt.grid(True)
plt.show()